# 2024 유로 스페인 빌드업 패턴: 패스 네트워크

2024 UEFA 유로에서 스페인이 치른 7경기(조별리그 3 + 토너먼트 4) 전체의 패스 네트워크를 그려 경기별로 빌드업이 어떻게 달랐는지, 대회를 관통하는 스타일은 무엇이었는지 살펴봅니다.

- 주 방법론은 **포지션(역할) 기반 통합 네트워크**입니다 - 노드를 선수 이름이 아니라 포지션 슬롯으로 잡아 교체와 무관하게 경기 전체(90분) 패스를 합산합니다. 오른쪽 패널에 포지션별 선수 로스터와 교체 시각을 함께 표시합니다.
- 선수 라벨은 `lineups`의 `player_nickname`을 사용합니다 (성이 두 개인 스페인 선수 표기 오류 방지).
- 이 폴더(`pass_network/`)는 "2024 유로 스페인의 빌드업 패턴" 주제의 패스 네트워크 방법론 전용 하위 폴더입니다. 분석 기획은 [`../PLAN.md`](../PLAN.md), 종합 결과는 [`RESULTS.md`](./RESULTS.md), 백로그 항목은 `ideas/backlog.md`의 "2024 유로 스페인의 빌드업 패턴"을 참고하세요.

## 방법론: 포지션 기반 통합 패스 네트워크를 어떻게 도출하는가

`plot_pass_network_by_position()`(`src/visualizer.py`)의 계산 순서는 다음과 같습니다.

1. **선수별 대표 포지션**: 각 선수가 그 경기에서 가장 자주 등장한 `position` 값(최빈값)을 대표 포지션으로 정합니다.
2. **성공한 패스만 포함**합니다 (`pass_outcome`이 `NaN`, `pass_recipient` 존재).
3. **패스 시작 선수 → 대표 포지션, 패스 수신 선수 → 대표 포지션**으로 매핑한 뒤, "포지션 슬롯" 단위로 노드 좌표(해당 슬롯을 거쳐간 모든 선수의 평균 위치)와 슬롯 쌍 연결(엣지)을 집계합니다. 교체로 선수가 바뀌어도 같은 슬롯이면 계속 합산되므로 경기 전체(90분) 표본을 다 씁니다.
4. **오른쪽 패널**에 포지션별 선수 로스터(등장 순서 + 첫 등장 시각)를 표시해, 노드만으로는 안 보이는 "누가 언제 그 슬롯을 맡았는지"를 보완합니다.

**데이터 검토 결과** (7경기 전체, `position` 컬럼 결측률과 `Tactical Shift`·`Substitution` 빈도 - 착수 전 확인):

| 경기 | 결측률 | Tactical Shift | Substitution |
| :--- | ---: | ---: | ---: |
| GS vs Croatia | 0.5% | 1회 (67') | 5회 |
| GS vs Italy | 0.3% | 0회 | 5회 |
| GS vs Albania | 0.3% | 1회 (61') | 5회 |
| R16 vs Georgia | 0.2% | 1회 (75') | 5회 |
| QF vs Germany (연장) | 0.5% | 2회 (79', 102') | 6회 |
| SF vs France | 0.5% | 2회 (57', 93') | 5회 |
| Final vs England | 0.3% | 1회 (89') | 4회 |

`position` 결측률은 모든 경기에서 1% 미만으로 안정적입니다. 다만 `Tactical Shift`가 있는 경기(7경기 중 5경기)는 포지션 라벨이 도중에 바뀔 수 있어, 오른쪽 패널의 로스터 상 "교체 쌍"이 실제 `Substitution` 이벤트의 OUT/IN과 정확히 일치하지 않을 수 있습니다(결승전 88~92분 구간에서 실제로 관찰됨). 이 점을 감안해 로스터 패널은 "그 슬롯을 누가 언제부터 맡았는지"의 근사치로 해석하세요.

**보조 방법론**: 특정 시점의 "실제 11명"을 보고 싶을 때는 `plot_pass_network()`(선수 이름 기준, `minute_range`로 구간 지정)를 보조적으로 사용합니다. 다만 구간이 짧으면(15분 미만) 표본 부족으로 품질이 떨어집니다.

In [ ]:
import os
import sys

if sys.platform.startswith('win'):
    sys.stdout.reconfigure(encoding='utf-8')

sys.path.append(os.path.dirname(os.path.dirname(os.getcwd())))

import matplotlib.pyplot as plt
from src.data_loader import get_competition_matches, get_match_events, get_match_lineups
from src.visualizer import plot_pass_network_by_position

COMPETITION_ID = 55  # UEFA Euro
SEASON_ID = 282      # 2024
TEAM = "Spain"

output_dir = os.path.join(os.getcwd(), "processed", "spain_euro2024_pass_networks")
os.makedirs(output_dir, exist_ok=True)

In [ ]:
matches = get_competition_matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)
spain_matches = matches[(matches['home_team'] == TEAM) | (matches['away_team'] == TEAM)].copy()
spain_matches = spain_matches.sort_values('match_date')
spain_matches[['match_id', 'match_date', 'home_team', 'away_team', 'home_score', 'away_score', 'competition_stage']]

In [ ]:
for _, match in spain_matches.iterrows():
    match_id = match['match_id']
    opponent = match['away_team'] if match['home_team'] == TEAM else match['home_team']
    stage = match['competition_stage']

    events = get_match_events(match_id=match_id)
    lineup = get_match_lineups(match_id=match_id)[TEAM]

    fig, axes = plot_pass_network_by_position(
        events_df=events,
        team_name=TEAM,
        lineup_df=lineup,
        title=f"Spain Pass Network by Position - {stage} vs {opponent}",
    )

    filename = f"{stage.lower().replace(' ', '_')}_vs_{opponent.lower().replace(' ', '_')}.png"
    out_path = os.path.join(output_dir, filename)
    fig.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='#1e1e1e')
    plt.show()
    plt.close(fig)
    print(f"저장 완료: {out_path}")

## 관찰 기록

7경기 결과를 보며 경기별 차이와 대회 전체를 관통하는 스페인 빌드업 스타일을 정리한 결과는 [`RESULTS.md`](./RESULTS.md)에 문서화했습니다.